In [1]:
# -*- coding: utf-8 -*-

# Phase 4 — 결정요인 모델링 (v4)

**v3에서 통째로 갈아탔다.** 기존 종속변수 `자동조정_허용`이 v4에 없고, 전제였던
"품목이 차종을 100% 결정(설명력 1.000)"도 v4에서는 0.444라 성립하지 않는다.

v4에는 경매 결과가 있으므로 **결과를 종속변수로 삼는다.** 세 개다.

| 절 | 종속변수 | 묻는 것 |
|---|---|---|
| 4-1 | `유찰` (0/1) | 어떤 조건이 배차 실패를 만드는가 |
| 4-2 | `log(체결배율)` | 어떤 조건이 운임을 밀어올리는가 |
| 4-3 | 제안 `수락` (0/1) | 화주는 어떤 제안을 받아들이는가 |

4-1·4-2는 **콜 단위**, 4-3은 **제안 단위**(개입제안로그)다.

---
**설계 판단 4가지 — 모델 적합 전에 데이터가 강제한 것**

① **개입 수락 콜은 결과가 덮어써졌다.** 제안을 수락한 408건은 조건이 바뀐 뒤 재경매한
   결과가 `최종체결운임`·`결과`에 들어 있다. 원래 조건으로 회귀하면 인과가 뒤집힌다.
   → 4-1·4-2는 **개입이 적용되지 않은 콜만** 쓴다.

② **`수락가능_현조건`은 공변량이 아니라 매개변수다.** 조건 → 후보 수 → 유찰이 생성
   메커니즘 그 자체라, 넣으면 다른 계수가 전부 0으로 눌린다. 기본 모델에서 빼고,
   매개 확인용으로 따로 한 번 넣어 비교한다.

③ **화주ID를 범주형으로 넣지 않는다.** 화주가 품목을 강하게 예측해 우회 경로가 된다.
   Phase 3의 세그먼트와 발주건수로 인코딩한다.

④ **차종을 통째로 넣지 않는다.** `적재형태` + `톤급`으로 분해해야 톤급 때문인지
   형태 때문인지 갈린다.

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.inspection import permutation_importance
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

for _f in ["Noto Sans CJK KR", "Noto Sans CJK JP", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
    if any(_f in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False

load_dotenv()
SRC = Path(os.getenv("DATA_DIR", ".")).expanduser().resolve()
OUT = SRC / "out"; OUT.mkdir(exist_ok=True)
FIG = OUT / "fig"; FIG.mkdir(exist_ok=True)

_pv = OUT / "_provenance.json"
if _pv.exists():
    _p = json.loads(_pv.read_text(encoding="utf-8"))
    print(f"[출처] {_p['source']} — 콜 {_p['rows']}행 / 제안 {_p['제안로그']}행 / 화주 {_p['화주수']}명")

df = pd.read_csv(OUT / "phase1_분석테이블.csv", parse_dates=["등록일시"])
prop = pd.read_csv(OUT / "phase1_제안로그.csv")
seg = pd.read_csv(OUT / "phase3_화주세그먼트.csv", index_col=0)

df = df.merge(seg[["세그먼트_규칙", "발주건수"]], left_on="화주ID", right_index=True, how="left")
assert df["세그먼트_규칙"].notna().all(), "세그먼트 병합 실패 — phase3 먼저 실행"
print(f"[load] 콜 {len(df)} / 제안 {len(prop)}")

# ── 희소 세그먼트 병합 ─────────────────────────────────────────
# 격주 세그먼트는 화주가 1~2명이라 콜이 10건도 안 된다. 더미로 넣으면 유찰이 전부 0이거나
# 1이 되어(완전분리) 계수가 ±30으로 발산하고 표준오차가 1e6까지 튄다.
# 진단만 찍고 넘어가면 안 되므로 적합 전에 흡수한다.
MIN_SEG = 30
_cnt = df["세그먼트_규칙"].value_counts()
_rare = list(_cnt[_cnt < MIN_SEG].index)
if _rare:
    print(f"[세그먼트] 희소 범주 {_rare} (각 {list(_cnt[_rare])}건) → '스팟'으로 흡수")
    df["세그먼트_규칙"] = df["세그먼트_규칙"].replace({r: "스팟" for r in _rare})
print(f"[세그먼트] 사용 범주: {dict(df['세그먼트_규칙'].value_counts())}")

pure = df[df["개입수락"] == 0].copy()
print(f"[표본] 개입 미적용 콜 {len(pure)}건 (전체 {len(df)} − 개입수락 {int(df['개입수락'].sum())})")


def design(d):
    """설계행렬 — 차종은 적재형태+톤급으로 분해, 화주는 세그먼트로 인코딩

    ⚠ `유연성지수(_코어)`를 통째로 넣지 않는다. 그 안에 `flex_time = 시간창/240`이
    들어 있어 `log_시간창`과 공선이 되고, 시간창 계수가 지수 쪽으로 흡수돼 0으로 눌린다.
    (실제로 그렇게 만들었다가 phase2 기술통계와 부호·크기가 어긋나 잡았다.)
    → 시간은 log_시간창, 차량은 대체허용, 화물은 flex_cargo로 **성분을 쪼개 넣는다.**
    """
    X = pd.DataFrame(index=d.index)
    X["log_시간창"] = np.log1p(d["시간창_분"])
    X["log_리드타임"] = np.log(d["리드타임_h"].clip(lower=1))
    X["log_거리"] = np.log(d["거리km"])
    X["톤급"] = d["톤급"]
    X["적재율"] = d["적재율"]
    X["대체허용"] = (d["차량유연성"] == "대체허용").astype(int)
    X["화물유연성"] = d["flex_cargo"]          # 분할·동시적재·경유·순서변경 4개 평균
    X["긴급"] = d["긴급플래그"]
    X["권한_미승인"] = d["권한_미승인"]
    X["발주건수"] = d["발주건수"]
    X = X.join(pd.get_dummies(d["적재형태"], prefix="형태", drop_first=True).astype(float))
    X = X.join(pd.get_dummies(d["세그먼트_규칙"], prefix="세그", drop_first=True).astype(float))
    return X.astype(float)


def vif_report(X, label, top=6):
    """분산팽창계수 — 10 넘으면 계수 해석 금지"""
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    Xc = sm.add_constant(X)
    v = pd.Series([variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])],
                  index=Xc.columns).drop("const").sort_values(ascending=False)
    bad = v[v > 10]
    print(f"  [VIF {label}] 최대 {v.iloc[0]:.1f} ({v.index[0]})"
          + (f"  ★ 10 초과: {list(bad.index)}" if len(bad) else "  — 이상 없음"))
    return v


def fit_logit(X, y, label):
    m = sm.Logit(y, sm.add_constant(X)).fit(disp=0, maxiter=200)
    r = pd.DataFrame({"계수": m.params, "표준오차": m.bse, "z": m.tvalues,
                      "p": m.pvalues, "오즈비": np.exp(m.params)})
    r["유의"] = np.where(r["p"] < .001, "***", np.where(r["p"] < .01, "**",
                                                     np.where(r["p"] < .05, "*", "")))
    print(f"\n[{label}] n={len(y)}, 사건={int(y.sum())}, "
          f"Pseudo R²={m.prsquared:.3f}, AUC={roc_auc_score(y, m.predict()):.3f}")
    print(r.round(4).to_string())
    return m, r

[출처] 유연오더_가상데이터_v13.xlsx — 콜 12000행 / 제안 9082행 / 화주 300명
[load] 콜 12000 / 제안 9082
[세그먼트] 사용 범주: {'고정주간': 7790, '스팟': 2587, '고빈도': 1526, '격주': 97}
[표본] 개입 미적용 콜 10773건 (전체 12000 − 개입수락 1227)


Matplotlib created a temporary cache directory at /var/folders/bc/xfffxrtd1yd5_w7skl8fpzzw0000gn/T/matplotlib-ztr6lidp because the default path (/Users/namkiwoong/.matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


## 4-1. 유찰 예측 — 어떤 조건이 배차 실패를 만드는가

In [3]:
print("\n" + "=" * 64 + "\n4-1. 유찰 모델\n" + "=" * 64)

print("[4-1-a] 완전분리 진단")
for c in ["긴급여부", "적재형태", "세그먼트_규칙", "등록주체", "차량유연성"]:
    t = pure.groupby(c)["유찰"].agg(["size", "mean"])
    ext = t[(t["mean"] == 0) | (t["mean"] == 1)]
    if len(ext):
        print(f"   ★ {c}: 완전분리 범주 {list(ext.index)} (n={list(ext['size'])})")
    else:
        print(f"     {c}: 이상 없음 (유찰률 {t['mean'].min():.3f}~{t['mean'].max():.3f})")

X1 = design(pure)
y1 = pure["유찰"]
vif_report(X1, "4-1")
m1, r1 = fit_logit(X1, y1, "4-1 유찰 (매개변수 제외)")
r1.round(4).to_csv(OUT / "phase4_유찰모델_계수.csv", encoding="utf-8-sig")

# ── 매개 확인: 후보 수를 넣으면 조건 효과가 사라지는가 ─────────────
X1m = design(pure)
X1m["log_후보수"] = np.log1p(pure["수락가능_현조건"])
m1m, r1m = fit_logit(X1m, y1, "4-1' 유찰 (후보 수 투입 — 매개 확인)")
print("\n[매개 판정] log_시간창 계수 변화: "
      f"{r1.loc['log_시간창','계수']:+.3f} → {r1m.loc['log_시간창','계수']:+.3f}")
print("   0에 가깝게 죽으면 '시간창 → 후보 수 → 유찰' 경로가 전부라는 뜻이다.")
print("   즉 시간창은 후보 수를 통해서만 작동한다 — 제품이 시간창을 여는 이유가 이것.")


4-1. 유찰 모델
[4-1-a] 완전분리 진단
     긴급여부: 이상 없음 (유찰률 0.091~0.141)
     적재형태: 이상 없음 (유찰률 0.084~0.114)
     세그먼트_규칙: 이상 없음 (유찰률 0.057~0.105)
     등록주체: 이상 없음 (유찰률 0.085~0.101)
     차량유연성: 이상 없음 (유찰률 0.088~0.100)
  [VIF 4-1] 최대 29.4 (세그_고정주간)  ★ 10 초과: ['세그_고정주간', '세그_스팟', '세그_고빈도']

[4-1 유찰 (매개변수 제외)] n=10773, 사건=1050, Pseudo R²=0.119, AUC=0.757
              계수    표준오차        z       p       오즈비   유의
const     5.1059  0.6682   7.6418  0.0000  164.9932  ***
log_시간창  -0.0909  0.0141  -6.4670  0.0000    0.9131  ***
log_리드타임 -2.1645  0.1010 -21.4288  0.0000    0.1148  ***
log_거리   -0.0692  0.0432  -1.6033  0.1089    0.9331     
톤급        0.0017  0.0061   0.2723  0.7854    1.0017     
적재율       0.4987  0.2782   1.7927  0.0730    1.6466     
대체허용     -0.0587  0.0881  -0.6656  0.5057    0.9430     
화물유연성    -0.0270  0.1771  -0.1527  0.8787    0.9733     
긴급       -2.8179  0.1802 -15.6338  0.0000    0.0597  ***
권한_미승인    0.0383  0.1079   0.3546  0.7229    1.0390     
발주건수     -0.0148  0.0063  -2.3

## 4-2. 체결배율 회귀 — 어떤 조건이 운임을 밀어올리는가

성사 건만 대상이다(유찰은 체결운임이 없다). **선택편향이 생긴다** — 조건이 나쁜 건은
유찰로 빠지므로 남은 표본은 상대적으로 조건이 나은 쪽에 치우친다.
그래서 계수는 "성사한 건들 사이에서의 효과"로만 읽어야 한다.

In [4]:
print("\n" + "=" * 64 + "\n4-2. 체결배율 모델\n" + "=" * 64)

ok = pure[pure["결과"] == "성사"].copy()
X2 = design(ok)
y2 = np.log(ok["체결배율"])
m2 = sm.OLS(y2, sm.add_constant(X2)).fit(cov_type="HC3")   # 이분산 강건 표준오차
r2 = pd.DataFrame({"계수": m2.params, "표준오차": m2.bse, "t": m2.tvalues, "p": m2.pvalues})
r2["운임효과%"] = (np.exp(m2.params) - 1) * 100
r2["유의"] = np.where(r2["p"] < .001, "***", np.where(r2["p"] < .01, "**",
                                                   np.where(r2["p"] < .05, "*", "")))
print(f"[4-2 log(체결배율)] n={len(y2)}, R²={m2.rsquared:.3f}, adj={m2.rsquared_adj:.3f}")
print(r2.round(4).to_string())
r2.round(4).to_csv(OUT / "phase4_운임모델_계수.csv", encoding="utf-8-sig")

print(f"\n[해석] 시간창 10배 → 체결배율 "
      f"{(np.exp(r2.loc['log_시간창','계수'] * np.log(10)) - 1) * 100:+.1f}%")
print(f"       긴급 건    → 체결배율 {r2.loc['긴급','운임효과%']:+.1f}%")


4-2. 체결배율 모델
[4-2 log(체결배율)] n=9723, R²=0.328, adj=0.327
              계수    표준오차        t       p    운임효과%   유의
const     0.2664  0.0195  13.6711  0.0000  30.5308  ***
log_시간창  -0.0068  0.0004 -15.5684  0.0000  -0.6770  ***
log_리드타임 -0.0403  0.0023 -17.7996  0.0000  -3.9493  ***
log_거리   -0.0018  0.0016  -1.1435  0.2528  -0.1805     
톤급        0.0005  0.0002   2.2588  0.0239   0.0461    *
적재율       0.0044  0.0091   0.4839  0.6285   0.4415     
대체허용     -0.0096  0.0028  -3.3781  0.0007  -0.9568  ***
화물유연성     0.0017  0.0058   0.2859  0.7750   0.1662     
긴급        0.1620  0.0068  23.8538  0.0000  17.5865  ***
권한_미승인   -0.0177  0.0041  -4.3317  0.0000  -1.7546  ***
발주건수     -0.0001  0.0002  -0.4956  0.6202  -0.0112     
형태_냉장     0.0110  0.0080   1.3688  0.1711   1.1027     
형태_윙바디   -0.0023  0.0036  -0.6449  0.5190  -0.2291     
형태_카고    -0.0048  0.0037  -1.2872  0.1980  -0.4744     
형태_탑차     0.0002  0.0038   0.0622  0.9504   0.0237     
세그_고빈도    0.0138  0.0122   1.1280  0.2593   1.

## 4-3. 제안 수락 모델 — 화주는 어떤 제안을 받아들이는가

단위가 콜이 아니라 **제안**이다. 한 콜에 최대 4개 제안이 달리므로(v5, 레버 6종) 같은
콜의 제안끼리는 독립이 아니다. 콜 단위 군집 강건표준오차로 보정한다.

**`제안유형`을 더미로 넣은 단일 모델**로 간다. 레버별로 모델을 6개 만들면 표본이
얇은 레버는 적합 자체가 안 된다. 이 설계는 Phase 5에서도 그대로 쓴다.

레버 중 하나라도 수락률이 정확히 0%(또는 100%)면 그 더미가 결과를 완벽하게 갈라버려
로지스틱 MLE가 발산한다 — 위 4-1·4-2 앞의 희소 세그먼트 흡수와 같은 문제라 같은
원칙으로 처리한다(적합 전에 제외).

In [5]:
print("\n" + "=" * 64 + "\n4-3. 제안 수락 모델\n" + "=" * 64)

P = prop.merge(
    df[["콜ID", "긴급플래그", "flex_cargo", "리드타임_h", "시간창_분", "톤급",
        "거리km", "세그먼트_규칙", "권한_미승인"]], on="콜ID", how="left")
P["수락"] = (P["사용자반응"] == "수락").astype(int)
print(f"[표본] 제안 {len(P)}건 / 수락 {int(P['수락'].sum())}건 ({P['수락'].mean()*100:.1f}%)")
print(P.groupby("제안유형")["수락"].agg(["size", "sum", "mean"]).round(3).to_string())

# ⚠ 완전분리 진단 — 어떤 레버가 수락률 0%(또는 100%)면 그 더미가 결과를 완벽하게
# 갈라버려 로지스틱 MLE가 수렴하지 않는다. 계수가 ±80대로 발산하고 표준오차는
# 반대로 작아 보여 z가 수백대까지 튄다 — 위 세그먼트 완전분리와 같은 현상이다.
# 진단만 찍고 넘어가면 안 되므로(위 MIN_SEG 흡수와 같은 원칙) 적합 전에 제외한다.
_lv_rate = P.groupby("제안유형")["수락"].agg(["size", "mean"])
_degenerate = list(_lv_rate[(_lv_rate["mean"] == 0) | (_lv_rate["mean"] == 1)].index)
if _degenerate:
    print(f"\n  ★ 완전분리 레버 {_degenerate} "
          f"(각 수락률 {list(_lv_rate.loc[_degenerate, 'mean'])}) → 이 회귀에서 제외")
    print("    (표본 자체가 학습 불가 수준으로 작다는 뜻이지, 레버가 극단적으로 나쁘다는")
    print("     뜻이 아니다 — 발산한 계수를 그대로 읽으면 안 된다)")
    P = P[~P["제안유형"].isin(_degenerate)].copy()

XP = pd.DataFrame(index=P.index)
XP["신뢰도"] = P["신뢰도"]
XP["제안순위"] = P["제안순위"]
XP["긴급"] = P["긴급플래그"]
XP["화물유연성"] = P["flex_cargo"]
XP["log_리드타임"] = np.log(P["리드타임_h"].clip(lower=1))
XP["log_시간창"] = np.log1p(P["시간창_분"])
XP = XP.join(pd.get_dummies(P["제안유형"], prefix="레버", drop_first=True).astype(float))
XP = XP.join(pd.get_dummies(P["세그먼트_규칙"], prefix="세그", drop_first=True).astype(float))
XP = XP.astype(float)

# ⚠ 제안의 예측치(예측_수락가능·예측_배차분)는 레버가 결정한다. 레버 더미와 같이 넣으면
# 거의 공선이 되어 레버 계수의 표준오차가 폭발한다(실제로 SE가 1.8~2.1까지 갔다).
# 레버 효과를 보려는 절이므로 예측치를 빼고, 예측치의 기여는 Phase 5 예측 모델에서 본다.
vif_report(XP, "4-3")

mP = sm.Logit(P["수락"], sm.add_constant(XP)).fit(
    disp=0, cov_type="cluster", cov_kwds={"groups": P["콜ID"]}, maxiter=200)
rP = pd.DataFrame({"계수": mP.params, "표준오차": mP.bse, "z": mP.tvalues,
                   "p": mP.pvalues, "오즈비": np.exp(mP.params)})
rP["유의"] = np.where(rP["p"] < .001, "***", np.where(rP["p"] < .01, "**",
                                                   np.where(rP["p"] < .05, "*", "")))
print(f"\n[4-3 제안 수락] n={len(P)}, 수락={int(P['수락'].sum())}, "
      f"Pseudo R²={mP.prsquared:.3f}, AUC={roc_auc_score(P['수락'], mP.predict()):.3f}")
print("  (콜 단위 군집 강건 표준오차)")
print(rP.round(4).to_string())
rP.round(4).to_csv(OUT / "phase4_수락모델_계수.csv", encoding="utf-8-sig")

_base = sorted(P["제안유형"].unique())[0]
print(f"\n[레버 오즈비 — 기준: {_base}]")
for c in [c for c in rP.index if c.startswith("레버_")]:
    print(f"   {c:12s} OR={rP.loc[c,'오즈비']:.3f}  p={rP.loc[c,'p']:.4f} {rP.loc[c,'유의']}")


4-3. 제안 수락 모델
[표본] 제안 9082건 / 수락 1354건 (14.9%)
      size  sum   mean
제안유형                  
가격     539   70  0.130
권한     453   65  0.143
날짜    1254  121  0.096
분할     383   28  0.073
시간    3422  680  0.199
차종    3031  390  0.129
  [VIF 4-3] 최대 34.4 (세그_고정주간)  ★ 10 초과: ['세그_고정주간', '세그_스팟', '세그_고빈도', '레버_시간']

[4-3 제안 수락] n=9082, 수락=1354, Pseudo R²=0.022, AUC=0.608
  (콜 단위 군집 강건 표준오차)
              계수    표준오차       z       p     오즈비   유의
const    -0.5631  0.5846 -0.9633  0.3354  0.5694     
신뢰도      -0.2483  0.3700 -0.6711  0.5021  0.7801     
제안순위     -0.2595  0.0904 -2.8725  0.0041  0.7714   **
긴급       -0.0205  0.1041 -0.1966  0.8441  0.9797     
화물유연성     0.4723  0.1200  3.9351  0.0001  1.6036  ***
log_리드타임  0.0088  0.0541  0.1624  0.8710  1.0088     
log_시간창  -0.0633  0.0115 -5.4897  0.0000  0.9387  ***
레버_권한    -0.7065  0.3058 -2.3105  0.0209  0.4933    *
레버_날짜    -0.7925  0.2250 -3.5229  0.0004  0.4527  ***
레버_분할    -0.9793  0.2661 -3.6807  0.0002  0.3756  ***
레버_시간    -0.2240 

## 4-4. GBM 순열중요도 — 반드시 홀드아웃에서 계산

로지스틱은 선형·가법을 가정한다. 비선형 상호작용이 있는지 GBM으로 교차 확인하고,
중요도는 **훈련셋이 아니라 홀드아웃**에서 순열로 잰다(훈련셋 중요도는 과적합을 반영).

In [6]:
print("\n" + "=" * 64 + "\n4-4. GBM 교차 확인\n" + "=" * 64)

for name, Xg, yg in [("유찰", X1, y1), ("제안수락", XP, P["수락"])]:
    Xtr, Xte, ytr, yte = train_test_split(Xg, yg, test_size=.3, random_state=42, stratify=yg)
    g = GradientBoostingClassifier(random_state=42, n_estimators=200, max_depth=3).fit(Xtr, ytr)
    pr = g.predict_proba(Xte)[:, 1]
    print(f"\n[{name}] 홀드아웃 AUC={roc_auc_score(yte, pr):.3f}  "
          f"Brier={brier_score_loss(yte, pr):.4f}  (기저율 {yte.mean():.3f})")
    pi = permutation_importance(g, Xte, yte, n_repeats=20, random_state=42, scoring="roc_auc")
    imp = pd.Series(pi.importances_mean, index=Xg.columns).sort_values(ascending=False)
    print(imp.head(8).round(4).to_string())
    imp.round(5).to_frame("중요도_홀드아웃").to_csv(
        OUT / f"phase4_GBM_중요도_{name}.csv", encoding="utf-8-sig")


4-4. GBM 교차 확인

[유찰] 홀드아웃 AUC=0.760  Brier=0.0834  (기저율 0.097)
log_리드타임    0.2147
log_시간창     0.0512
긴급          0.0194
적재율         0.0085
발주건수        0.0039
톤급          0.0021
형태_카고       0.0019
대체허용        0.0009

[제안수락] 홀드아웃 AUC=0.597  Brier=0.1264  (기저율 0.149)
log_시간창    0.0612
레버_시간      0.0379
제안순위       0.0239
레버_날짜      0.0085
신뢰도        0.0052
레버_분할      0.0049
레버_차종      0.0023
세그_고정주간    0.0012


## 시각화

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

_c = r1.drop("const"); _c = _c.reindex(_c["계수"].abs().sort_values().index)
ax[0].barh(_c.index, _c["계수"], color=np.where(_c["계수"] > 0, "#C25E5E", "#3B6EA5"))
ax[0].axvline(0, c="k", lw=.8); ax[0].set_title("4-1 유찰 로지스틱 계수", fontsize=12)

_c2 = r2.drop("const"); _c2 = _c2.reindex(_c2["계수"].abs().sort_values().index)
ax[1].barh(_c2.index, _c2["운임효과%"], color=np.where(_c2["계수"] > 0, "#C25E5E", "#3B6EA5"))
ax[1].axvline(0, c="k", lw=.8); ax[1].set_title("4-2 체결배율 효과(%)", fontsize=12)

_acc = P.groupby("제안유형")["수락"].mean().sort_values(ascending=False)
ax[2].bar(_acc.index, _acc.values * 100, color="#5B8C5A")
ax[2].set_title("4-3 레버별 실측 수락률(%)", fontsize=12)

plt.tight_layout()
plt.savefig(FIG / "phase4_요약.png", dpi=130, bbox_inches="tight")
print(f"\n[저장] {OUT}")


[저장] /Users/namkiwoong/Documents/Codex/2026-08-12/new-chat-5/work/hackathon_analysis/out
